# Alpamayo-v1.5 KV-Cache Bit-Flip Attack Demo (Phase 2a)

Targets the **VLM prefill K/V cache** (HF `DynamicCache`) rather than weights.
This is the activation-level twin of the Phase-1 weight BFA in `bfa_demo.ipynb`.

Pipeline per trial:

1. Look up `ctx.prompt_cache.key_cache[l]` (or `.value_cache[l]`) — **re-fetched each trial** because `DynamicCache` replaces these list entries during the cat/crop cycle.
2. `bf16_flip_one` at a gradient-ranked flat index.
3. Re-run `fm_one_step_loss`; record `post_loss`.
4. `restore_one` — always, via try/finally.

Gradients (`∂loss/∂K_l`, `∂loss/∂V_l`) come from **one expert-only backward**.
The VLM is never re-run; no FSDP required.

See `docs/sdc/alpamayo_bfa_experiments.md` §10 for the full spec.

Output: `bfa_kv_results.json` with per-trial `{layer_idx, kind, bit, flat_idx, delta_loss, ...}`.

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

from alpamayo1_5.models.alpamayo1_5 import Alpamayo1_5
from alpamayo1_5.load_physical_aiavdataset import load_physical_aiavdataset
from alpamayo1_5 import helper

import bfa_utils as bfa        # sits next to this notebook

In [ ]:
model = Alpamayo1_5.from_pretrained("nvidia/Alpamayo-1.5-10B", dtype=torch.bfloat16).to("cuda")
model.eval()
# Expert/projection params need grad for the FM backward to reach them —
# from_pretrained usually leaves requires_grad=True, but be explicit.
for p in model.parameters():
    p.requires_grad_(True)

processor = helper.get_processor(model.tokenizer)

clip_ids = pd.read_parquet("clip_ids.parquet")["clip_id"].tolist()
clip_id = clip_ids[774]   # same default as inference.ipynb and bfa_demo.ipynb
data = load_physical_aiavdataset(clip_id)

messages = helper.create_message(
    data["image_frames"].flatten(0, 1),
    camera_indices=data["camera_indices"],
)
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False,
    continue_final_message=True,
    return_dict=True,
    return_tensors="pt",
)
model_inputs = helper.to_device(
    {
        "tokenized_data": inputs,
        "ego_history_xyz": data["ego_history_xyz"],
        "ego_history_rot": data["ego_history_rot"],
    },
    "cuda",
)

In [ ]:
# GT action in the form the FM loss expects.
gt_action = model.action_space.traj_to_action(
    traj_history_xyz=data["ego_history_xyz"].cuda().to(torch.float32),
    traj_history_rot=data["ego_history_rot"].cuda().to(torch.float32),
    traj_future_xyz=data["ego_future_xyz"].cuda().to(torch.float32),
    traj_future_rot=data["ego_future_rot"].cuda().to(torch.float32),
)
gt_action = gt_action.view(-1, *model.action_space.get_action_space_dims())
print("gt_action:", gt_action.shape, gt_action.dtype)

with torch.autocast("cuda", dtype=torch.bfloat16):
    ctx = bfa.build_fm_context(model, model_inputs, gt_action)
print("prefill_seq_len:", ctx.prefill_seq_len)
print("n_kv_layers:", len(ctx.prompt_cache.key_cache))
print("K_0 shape:", tuple(ctx.prompt_cache.key_cache[0].shape),
      "dtype:", ctx.prompt_cache.key_cache[0].dtype)

# Fixed noise + fixed t_val so every trial uses the SAME x_t — only the
# cache perturbation varies across trials.
torch.cuda.manual_seed_all(42)
fixed_noise = torch.randn_like(ctx.gt_action)
T_VAL = 0.5

def loss_fn():
    with torch.autocast("cuda", dtype=torch.bfloat16):
        return bfa.fm_one_step_loss(model, ctx, t_val=T_VAL, noise=fixed_noise)

baseline = loss_fn().item()
print("baseline FM loss:", baseline)

In [ ]:
kv_targets = bfa.collect_kv_cache_targets(ctx)
print(f"{len(kv_targets)} KV targets ({len(kv_targets)//2} layers × K,V)")
print("first 4:", list(kv_targets.keys())[:4])

# One expert-only backward (no FSDP, VLM not re-run).
t0 = time.time()
kv_grads = bfa.compute_clean_kv_grads(model, ctx, kv_targets, loss_fn)
print(f"backward done in {time.time()-t0:.1f}s")

# Sanity on grad magnitudes: at least one target should have non-trivial grad.
grad_norms = {n: g.norm().item() for n, g in kv_grads.items()}
top5 = sorted(grad_norms.items(), key=lambda kv: -kv[1])[:5]
print("top-5 targets by |grad|:")
for n, gn in top5:
    print(f"  {n:24s}  |grad| = {gn:.4g}")

## Smoke tests

Run BEFORE the main loop. If any fail, investigate before spending time on the full sweep.

- **(a)** bit-flip primitive still works on a cache tensor (sanity).
- **(b)** flip → run `loss_fn` → restore round-trip returns loss to baseline.
- **(c)** `DynamicCache.crop` preserves prefix VALUES across calls (the load-bearing invariant for cross-trial independence).

In [ ]:
# (a) bit-flip primitive on a cache tensor
t = bfa.get_kv_tensor(ctx, layer_idx=0, kind="key")
assert t.dtype == torch.bfloat16, t.dtype
assert t.is_contiguous(), "expected contiguous cache tensor"

snapshot = t.clone()
orig, flipped = bfa.bf16_flip_one(t, flat_idx=123, bit=0)
assert not torch.equal(t, snapshot), "flip produced no change in cache tensor"
bfa.restore_one(t, 123, orig)
assert torch.equal(t, snapshot), "restore did not reproduce clean cache tensor"
print("(a) cache bit-flip primitive: OK")

In [ ]:
# (b) full flip → loss_fn → restore round trip returns cache values and loss
#     to their clean state. This is exactly what measure_kv_flipped_loss does
#     under the hood, tested explicitly here.
with torch.no_grad():
    l1 = loss_fn().item()
    l2 = loss_fn().item()
assert abs(l1 - l2) < 1e-5, (l1, l2)
print(f"baseline determinism: OK ({l1})")

# Bit 0 = mantissa LSB: should produce a small finite delta.
r_safe = bfa.measure_kv_flipped_loss(
    model, ctx, layer_idx=0, kind="key",
    flat_idx=123, bit=0, loss_fn=loss_fn,
)
print("bit-0 flip post_loss:", r_safe["post_loss"],
      "(finite:", bool(r_safe["post_loss_finite"]), ")")

# After restore, loss_fn must return exactly the baseline again.
with torch.no_grad():
    l3 = loss_fn().item()
assert abs(l3 - l1) < 1e-5, f"baseline not recovered: {l3} vs {l1}"
print("(b) flip-and-restore round trip: OK")

# Bit 14 = exponent MSB: catastrophic, should produce inf/nan loss.
r_cat = bfa.measure_kv_flipped_loss(
    model, ctx, layer_idx=0, kind="key",
    flat_idx=123, bit=14, loss_fn=loss_fn,
)
print("bit-14 flip post_loss:", r_cat["post_loss"],
      "(finite:", bool(r_cat["post_loss_finite"]), ")")
with torch.no_grad():
    l4 = loss_fn().item()
assert abs(l4 - l1) < 1e-5, f"baseline not recovered after bit-14: {l4} vs {l1}"
print("    catastrophic bit also restored cleanly: OK")

In [ ]:
# (c) crop-invariant check — prefix VALUES must survive a loss_fn call.
#
# Even though DynamicCache replaces key_cache[l] with a new tensor object
# during the cat/crop cycle, the prefix slice should contain the same values
# as before. Verify by sampling the tensor at a prefix position, calling
# loss_fn, and re-sampling.
k_before = bfa.get_kv_tensor(ctx, 0, "key").clone()

with torch.no_grad():
    _ = loss_fn().item()   # runs expert forward + crop internally

k_after = bfa.get_kv_tensor(ctx, 0, "key")
assert k_after.shape == k_before.shape, (k_after.shape, k_before.shape)
diff = (k_after.float() - k_before.float()).abs().max().item()
print(f"max |ΔK_layer0| across loss_fn call: {diff:.3e}")
assert diff == 0.0, "prefix K values changed across a loss_fn call — crop invariant broken"

# Object-identity check (informative, not asserted): is the Python object
# still the same, or was it replaced?
same_object = k_after.data_ptr() == k_before.data_ptr()
print(f"storage pointer preserved: {same_object}")
if not same_object:
    print("    -> values preserved but tensor was replaced; get_kv_tensor"
          " re-fetch in measure_kv_flipped_loss is what makes this safe.")
print("(c) crop-invariant: OK")

In [ ]:
# (d) top-k-vs-random sanity (single sample). Not a strict assertion —
# we expect top-k to dominate in aggregate over the main loop.
bit = 14
name = "layer0.key"
layer_idx, kind = kv_targets[name]
t = bfa.get_kv_tensor(ctx, layer_idx, kind)
flat_idx, bg_vals = bfa.topk_bitflip_coords(t, kv_grads[name], bit=bit, k=1)
r_top = bfa.measure_kv_flipped_loss(
    model, ctx, layer_idx, kind,
    flat_idx=int(flat_idx[0].item()), bit=bit, loss_fn=loss_fn,
)

rng = np.random.default_rng(0)
random_idx = int(rng.integers(0, t.numel()))
r_rand = bfa.measure_kv_flipped_loss(
    model, ctx, layer_idx, kind,
    flat_idx=random_idx, bit=bit, loss_fn=loss_fn,
)

print(f"top-k delta  ({name}, bit {bit}): {r_top['post_loss'] - baseline:.4g}")
print(f"random delta ({name}, bit {bit}): {r_rand['post_loss'] - baseline:.4g}")

In [ ]:
# Main sweep
BITS = list(range(16))          # full BF16 sweep. Use [14, 15] for a quick check.
TOP_K_PER_TARGET = 20           # per (target, bit).
TARGET_SUBSET = list(kv_targets.keys())   # or filter to e.g. ["layer0.key", "layer17.value"]

kv_results = []
t_start = time.time()
for bit in BITS:
    for name in TARGET_SUBSET:
        layer_idx, kind = kv_targets[name]
        t = bfa.get_kv_tensor(ctx, layer_idx, kind)
        g = kv_grads[name]
        flat_idx, bg_vals = bfa.topk_bitflip_coords(t, g, bit=bit, k=TOP_K_PER_TARGET)
        for i, idx in enumerate(flat_idx.tolist()):
            r = bfa.measure_kv_flipped_loss(
                model, ctx, layer_idx, kind,
                flat_idx=idx, bit=bit, loss_fn=loss_fn,
            )
            r.update({
                "target": name,
                "rank": i,
                "predicted_bit_grad": float(bg_vals[i].item()),
                "baseline_loss": baseline,
                "delta_loss": r["post_loss"] - baseline,
            })
            kv_results.append(r)
    print(f"bit {bit} done, {len(kv_results)} trials, {time.time()-t_start:.0f}s elapsed")

out_path = Path("bfa_kv_results.json")
out_path.write_text(json.dumps(kv_results, indent=2))
print(f"saved {len(kv_results)} trials to {out_path}")

In [ ]:
df = pd.DataFrame(kv_results)
print("per-bit summary:")
print(df.groupby("bit")["delta_loss"].describe()[["count", "mean", "max"]])

print("\nper-kind summary (K vs V):")
print(df.groupby("kind")["delta_loss"].describe()[["count", "mean", "max"]])

df_finite = df[df["post_loss_finite"] == 1.0]
print("\nTop-10 single-bit cache flips by delta_loss (finite-loss only):")
print(df_finite.sort_values("delta_loss", ascending=False).head(10)[
    ["target", "layer_idx", "kind", "bit", "flat_idx",
     "orig_value", "flipped_value", "delta_loss"]
])
n_inf = int((df["post_loss_finite"] == 0.0).sum())
print(f"\n({n_inf} additional trials produced non-finite loss — see post_loss_finite column)")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df.boxplot(column="delta_loss", by="bit", ax=ax)
ax.set_yscale("symlog")
ax.set_title("KV-cache BFA: per-bit delta_loss (BF16, gradient-guided top-k)")
ax.set_xlabel("bit index (0 = mantissa LSB, 14 = exp MSB, 15 = sign)")
ax.set_ylabel("post_loss - baseline")
plt.suptitle("")
plt.tight_layout()
plt.savefig("bfa_kv_per_bit.png", dpi=120)
plt.show()

In [ ]:
# Per-layer view — unique to the KV experiment since each layer's K and V
# are distinct targets. Separate boxes for K vs V at each layer.
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for ax, kind in zip(axes, ["key", "value"]):
    sub = df[df["kind"] == kind]
    sub.boxplot(column="delta_loss", by="layer_idx", ax=ax)
    ax.set_yscale("symlog")
    ax.set_title(f"per-layer delta_loss, kind = {kind}")
    ax.set_ylabel("post_loss - baseline")
axes[-1].set_xlabel("layer index")
plt.suptitle("")
plt.tight_layout()
plt.savefig("bfa_kv_per_layer.png", dpi=120)
plt.show()